# **Интерпритация DeepCA**

## *Библиотеки*

In [ ]:
import torchvision.transforms as T
import numpy as np
import os
import torch
from common import *
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from Reconstruction_coronary_arteria.open_json import GeneratedDataset
import cv2
from scipy.ndimage import map_coordinates, zoom, label  # Добавляем импорт zoom
import PIL 


## **Воссоздание воксельной структуры обратного преобразования**

In [ ]:
# Пути к данным
images_dir = "/home/alexus/Desktop/For_Generator/data0/content/vessel_tree_generator/data/test/images/test/"
labels_dir = "/home/alexus/Desktop/For_Generator/data0/content/vessel_tree_generator/data/test/labels/test/"
info_dir = "/home/alexus/Desktop/For_Generator/data0/content/vessel_tree_generator/data/test/info/"

# Трансформации для проекций
def mean_channel(x):
    return x.mean(axis=0)[None, :]

def distance_transform(image_mask):
    image_mask = np.asarray(image_mask)[0]
    binary_mask = (image_mask > np.mean(image_mask)).astype(np.uint8)
    distance = cv2.distanceTransform(binary_mask, cv2.DIST_L2, 5)
    return torch.tensor(image_mask + distance[None, :])

# images_transform = T.Compose([
    # T.ToTensor(),
    # T.Lambda(lambda x: x.mean(axis=0)[None, :] if x.shape[0] > 1 else x),
    # T.Resize((128, 128)),
    # T.Lambda(lambda x: x * 5.0),  # Масштабируем обратно до 0.25–5.0
# ])
# 
# Загрузка данных
# data = GeneratedDataset(images_dir, labels_dir, info_dir, images_transform=images_transform)
# sample_index = 0 #  индекс проекции
# images, model, theta, phi = data[sample_index]


# Трансформация
images_transform = T.Compose([
    T.ToTensor(),
    T.Resize((128, 128)),
    T.Lambda(lambda x: x * 5.0),
])

data = GeneratedDataset(images_dir, labels_dir, info_dir, images_transform=images_transform)
sample_index = 18
sample = data[sample_index]



In [ ]:
test_list = [['x','y','z'],['x','y','z'],['x','y','z'],['x','y','z']]
len(test_list)

In [ ]:
"""
data - [index образца 0-19][0 - images, 1 - 3D model (300 points), 2 - theta, 3 - phi][index proj = 0-1]
"""
plt.figure(figsize=(15,15))
plt.subplot(1,4,1)
plt.imshow(sample[0][0])
plt.axis('off')
plt.subplot(1,4,2)
plt.imshow(sample[0][4])
plt.axis('off')

In [ ]:
class Backprojection:
    def __init__(self, sid, pixel_spacing, volume_size, volume_spacing, dso=None, img_dim=128):
        self.sid = sid
        self.pixel_spacing = pixel_spacing
        self.volume_size = volume_size
        self.volume_spacing = volume_spacing
        self.dso = dso if dso is not None else sid * 0.75
        self.img_dim = img_dim
        self.principal_point = np.array([img_dim/2, img_dim/2])
        self.distance_detector_to_iso = sid - self.dso
        self.coord_system_changer = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) #[0, -1, 0], [1, 0, 0], [0, 0, -1] перевод в правую ДСК

    def get_projection_matrix(self, theta, phi): # Преобразование координат (поворот) в плоскости проекции
        theta = float(theta)
        phi = float(phi)
        theta_rad = np.radians(theta)
        phi_rad = np.radians(phi)

        """
        C = Vector_proj_coord = [-y, x, -z] - Медицинские координаты (Левая ДСК) x,y,z -> -y, x, -z - НЕ РАБОТАЕТ! 
        ТЕПЕРЬ С - ЭТО ПДСК [x,y,z], Саша, разберись почему это так стало работать!!! 
        Rotation_AP1 = C @ Rotate_around_Z @ C^-1, 
        где С^1 - обратная матрица, Rotate_around_Z - Матрица поворота вокруг Z (theta)
        Тем самым получаем поворот вокруг Z, но не в текущей системе координат, а в другой системе, которая определяется матрицей С.

        Rotation_AP2 = C @ Rotate_around_X @ C^-1,
        где Rotate_around_X - Матрица поворота вокруг Х (phi)
        """
        Rotation_AP1 = self.coord_system_changer @ np.array([
            [np.cos(theta_rad), -np.sin(theta_rad), 0],
            [np.sin(theta_rad), np.cos(theta_rad), 0],
            [0, 0, 1]
        ]) @ np.linalg.inv(self.coord_system_changer)

        Rotation_AP2 = self.coord_system_changer @ np.array([
            [1, 0, 0],
            [0, np.cos(phi_rad), np.sin(phi_rad)],
            [0, -np.sin(phi_rad), np.cos(phi_rad)]
        ]) @ np.linalg.inv(self.coord_system_changer)

        Rotation_AP3 = np.eye(3) # ед. матрица 3х3
        R_total = Rotation_AP1 @ Rotation_AP2 @ Rotation_AP3

        v_sensor = R_total @ np.array([0, 0, self.distance_detector_to_iso]) # вектор указывающий на точку приёмника
        v_source = -v_sensor / self.distance_detector_to_iso * self.dso # вектор указывающий на точку источника
        print(f"Focal point (sensor): {v_sensor}")
        print(f"Focal point (source): {v_source}")

        z_axis = (v_sensor - v_source) / np.linalg.norm(v_sensor - v_source) # это направление луча от источника к сенсору, т.е. главная ось камеры (оптическая ось).
        # Нормализация делает z_axis единичным вектором.
        x_axis = np.array([1, 0, 0])
        if np.abs(np.dot(z_axis, x_axis)) > 0.99:
            x_axis = np.array([0, 1, 0]) # Если z_axis почти коллинеарен оси x, выбирается другая ось, чтобы избежать численных проблем при ортогонализации.
        x_axis = x_axis - np.dot(x_axis, z_axis) * z_axis # Ортогонализация: мы "вычёркиваем" проекцию x_axis на z_axis, чтобы получить вектор, перпендикулярный z_axis.
        x_axis = x_axis / np.linalg.norm(x_axis)
        y_axis = np.cross(z_axis, x_axis)
        y_axis = y_axis / np.linalg.norm(y_axis)

        R = np.vstack((x_axis, y_axis, z_axis)).T # это 3×3 матрица поворота из мировой системы координат → в систему камеры
        t = -v_source # вектор смещения (переноса камеры)
        Rt = np.hstack((R, t.reshape(3, 1)))

        focal_length = self.sid / self.pixel_spacing # Делим  расстояние от источника до детектора на размер пикселя — получаем фокусное расстояние в пикселях
        K = np.array([
            [focal_length, 0, self.principal_point[0]],
            [0, focal_length, self.principal_point[1]],
            [0, 0, 1]
        ])
        """
        K — матрица внутренних параметров камеры (intrinsics)
        Она определяет:фокусное расстояние и положение главной точки (оптического центра) в пикселях
        """
        P = K @ Rt # итоговая матрица камеры 3×4, которая объединяет внутренние и внешние параметры:
        return P, v_source, v_sensor, R


In [ ]:
def create_shadow_cone(projection, theta, phi, volume_shape=(128, 128, 128), sid=1200.0, pixel_spacing=1.4, dso=750.0, img_dim=128):
    D_x, D_y, D_z = volume_shape
    projection = projection.clone().detach()
    print(f"Projection shape: {projection.shape}, min/max: {projection.min().item()}, {projection.max().item()}")
    threshold = 1
    # Отладка: посмотрим, сколько пикселей действительно ненулевые
    print(f"Pixels > 0.5: {(projection > 0.5).sum().item()}")
    print(f"Pixels > 1.0: {(projection > 1.0).sum().item()}")

    # Бинарная маска для визуализации
    projection_binary = (projection > 1).float()
    plt.imshow(projection_binary.cpu().numpy(), cmap='gray')
    plt.title(f"Projection for theta={theta}, phi={phi}")
    plt.colorbar()
    plt.show()

    # Создаём воксельное пространство, расширяем по X и Y
    vol = torch.zeros(volume_shape, dtype=torch.float32, device=projection.device)
    volume_spacing = 0.35
    x = torch.linspace(-100.0, 100.0, D_x, device=projection.device)  # Расширяем X
    y = torch.linspace(-100.0, 100.0, D_y, device=projection.device)  # Расширяем Y
    z = torch.linspace(-200.0, 200.0, D_z, device=projection.device)  # Z от -200 до 200 мм

    # Получаем геометрию камеры
    backprojector = Backprojection(sid, pixel_spacing, volume_shape, volume_spacing, dso, img_dim)
    P, v_source, v_sensor, R = backprojector.get_projection_matrix(theta, phi)

    # Преобразуем в тензоры
    v_source = torch.tensor(v_source, dtype=torch.float32, device=projection.device)
    v_sensor = torch.tensor(v_sensor, dtype=torch.float32, device=projection.device)
    R = torch.tensor(R, dtype=torch.float32, device=projection.device)

    # Определяем координаты пикселей проекции на сенсоре
    u_coords, v_coords = torch.where(projection > 1)
    intensities = projection[u_coords, v_coords]
    print(f"Non-zero pixels in projection: {len(u_coords)}")

    # Преобразуем u, v в 3D-координаты на сенсоре
    u_physical = (u_coords.float() - img_dim/2) * pixel_spacing
    v_physical = (v_coords.float() - img_dim/2) * pixel_spacing
    sensor_points = torch.stack([
        u_physical,
        v_physical,
        torch.zeros_like(u_physical)
    ], dim=1)

    # Поворачиваем точки сенсора в глобальную систему координат
    sensor_points = (R @ sensor_points.T).T + v_sensor

    # Для каждой точки на сенсоре строим луч к источнику
    directions = v_source - sensor_points
    directions = directions / torch.norm(directions, dim=1, keepdim=True)

    # Проходим по лучу через воксельное пространство
    t_max = 3000.0  # Увеличиваем расстояние
    t_steps = torch.linspace(0, t_max, 3000, device=projection.device)  # Больше шагов

    for i in range(len(sensor_points)):
        points_along_ray = sensor_points[i] + directions[i] * t_steps[:, None]
        
        # Преобразуем точки в воксельные индексы
        x_idx = ((points_along_ray[:, 0] - (-100.0)) / (200.0 / (D_x - 1))).long()
        y_idx = ((points_along_ray[:, 1] - (-100.0)) / (200.0 / (D_y - 1))).long()
        z_idx = ((points_along_ray[:, 2] - (-200.0)) / (400.0 / (D_z - 1))).long()

        # Проверяем, что индексы в пределах объёма
        valid = (x_idx >= 0) & (x_idx < D_x) & (y_idx >= 0) & (y_idx < D_y) & (z_idx >= 0) & (z_idx < D_z)
        
        # Отладка
        if i == 0:
            print(f"Points in ray: {len(t_steps)}, Valid points: {valid.sum().item()}")
            print(f"Sample points along ray (first 5): {points_along_ray[:5]}")

        x_idx, y_idx, z_idx = x_idx[valid], y_idx[valid], z_idx[valid]

        # Заполняем воксели
        vol[x_idx, y_idx, z_idx] = intensities[i]

    print(f"Non-zero voxels in shadow: {(vol > 0).sum().item()}")
    return vol, v_source, v_sensor, sensor_points, directions, projection_binary

def visualize_angiograph(projection1, projection2, theta, phi, volume_shape=(128, 128, 128)):
    # Создаём тени для обеих проекций
    vol1, v_source1, v_sensor1, sensor_points1, directions1, proj_binary1 = create_shadow_cone(projection1, theta[0], phi[0], volume_shape)
    vol2, v_source2, v_sensor2, sensor_points2, directions2, proj_binary2 = create_shadow_cone(projection2, theta[1], phi[1], volume_shape)
    D_x, D_y, D_z = volume_shape
    # Пересечение теней
    backprojection = (vol1 > 0) & (vol2 > 0)
    backprojection = backprojection.float()
    print(f"Non-zero voxels in backprojection: {(backprojection > 0).sum().item()}")

    # Plotly 1: Сензоры, источники и лучи
    fig1 = go.Figure()

    # 1. Источники (v_source)
    fig1.add_trace(go.Scatter3d(
        x=[v_source1[0].item(), v_source2[0].item()],
        y=[v_source1[1].item(), v_source2[1].item()],
        z=[v_source1[2].item(), v_source2[2].item()],
        mode='markers',
        marker=dict(size=10, color='red'),
        name='Sources'
    ))

    # 2. Сензоры (v_sensor) как плоскости
    sensor_size = 128 * 1.4 / 2
    for i, (v_sensor, projection) in enumerate([(v_sensor1, proj_binary1), (v_sensor2, proj_binary2)]):
        x = [v_sensor[0].item() - sensor_size, v_sensor[0].item() + sensor_size]
        y = [v_sensor[1].item() - sensor_size, v_sensor[1].item() + sensor_size]
        z = [v_sensor[2].item(), v_sensor[2].item()]
        fig1.add_trace(go.Surface(
            x=x,
            y=y,
            z=np.array(z)[None, :],
            surfacecolor=projection.cpu().numpy(),
            colorscale='Gray',
            showscale=False,
            opacity=0.7,
            name=f'Sensor {i+1}'
        ))

    # 3. Лучи (ограничим до 50)
    num_rays = 200
    for i in range(min(num_rays, len(sensor_points1))):
        start = sensor_points1[i].cpu().numpy()
        end = v_source1.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='blue', width=2),
            name='Ray 1' if i == 0 else None,
            showlegend=(i == 0)
        ))
    for i in range(min(num_rays, len(sensor_points2))):
        start = sensor_points2[i].cpu().numpy()
        end = v_source2.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='green', width=2),
            name='Ray 2' if i == 0 else None,
            showlegend=(i == 0)
        ))

    fig1.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="Angiograph: Sensors, Sources, and Rays"
    )
    fig1.show()

    # Plotly 2: Только 3D-реконструкция
    fig2 = go.Figure()

    non_zero = torch.where(backprojection > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1][indices].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2][indices].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0
    else:
        x = non_zero[0].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0

    fig2.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=2, color='purple', opacity=0.5),
        name='Reconstruction'
    ))

    fig2.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="3D Reconstruction"
    )
    fig2.show()

projection1 = sample[0][0].cuda()
projection2 = sample[0][4].cuda()
theta = sample[2]
phi = [90 + sample[3][0], 90 + sample[3][1]]

print(f"Using angles: theta={theta}, phi={phi}")

visualize_angiograph(projection1, projection2, theta, phi, volume_shape=[128, 128, 128])

## **3D Model**

In [ ]:
import plotly.graph_objects as go
from typing import List
from scipy.ndimage import zoom
import torch.nn.functional as F
def create_output_model(output_3d_model: torch.Tensor, volume_shape: List[int]):
    output_3d_model = output_3d_model.clone()
    # Центрируем координаты, как в исходном коде
    mean_coords = output_3d_model[..., :3].mean(dim=(0, 1))
    output_3d_model[..., :3] -= mean_coords
    # Масштабируем для соответствия объёму
    scale = float(output_3d_model[..., :3].max() - output_3d_model[..., :3].min())
    output_3d_model[..., :3] /= scale if scale != 0.0 else 1.0
    output_3d_model[..., -1] /= scale if scale != 0.0 else 1.0
    # Отображаем на индексы объёма (0 до 127)
    output_3d_model[..., :3] = (output_3d_model[..., :3] * 100 + 64).int().clip(0, 127)  # Центр в 64

    T = torch.zeros(volume_shape, dtype=torch.float32)

    for group in range(output_3d_model.shape[0]):
        x = output_3d_model[group, :, 0].long()  # Преобразуем в long
        y = output_3d_model[group, :, 1].long()
        z = output_3d_model[group, :, 2].long()
        R = output_3d_model[group, :, 3] * 10
        for i in range(len(x)):
            radius = int((R[i] * 12).clamp(1, 10))
            x_grid, y_grid, z_grid = torch.meshgrid(
                torch.arange(-radius, radius + 1, dtype=torch.long),
                torch.arange(-radius, radius + 1, dtype=torch.long),
                torch.arange(-radius, radius + 1, dtype=torch.long)
            )
            mask = (x_grid ** 2 + y_grid ** 2 + z_grid ** 2 <= radius ** 2)
            # Гарантируем, что результат операций остаётся в типе long
            x_shifted = (x[i] + x_grid[mask]).long()
            y_shifted = (y[i] + y_grid[mask]).long()
            z_shifted = (z[i] + z_grid[mask]).long()
            
            # Проверяем, что координаты находятся в пределах объёма
            valid = (x_shifted >= 0) & (x_shifted < 128) & (y_shifted >= 0) & (y_shifted < 128) & (z_shifted >= 0) & (z_shifted < 128)
            x_valid = x_shifted[valid]
            y_valid = y_shifted[valid]
            z_valid = z_shifted[valid]

            # Индексация тензора T
            insertion_index = T[x_valid, y_valid, z_valid] == 0
            average_index = T[x_valid, y_valid, z_valid] != 0

            T[x_valid, y_valid, z_valid] += R[i] * insertion_index
            avg = (R[i] + T[x_valid, y_valid, z_valid]) / 2
            T[x_valid, y_valid, z_valid] += (avg - T[x_valid, y_valid, z_valid]) * average_index

    return T #(4,300*6N,)

def get_3d_tensors(projection1,projection2,correct_3d_geometry  ,tensor_output_shape=[128,128,128]):
    """
    Из двух двумерных проекций `projection1` (1,128,128), `projection2` (1,128,128) строит трехмерный тензор размером `tensor_output_shape` который содержит в себе тени от
    двумерных проекций.

    Из информации о ветках `correct_3d_geometry` (4,300,4) строит 3д тензор

    Возвращает два 3д тенозора размером `volume_shape`, первый из которых - тензор теней, второй - тензор с 3д геометрией
    """

    #phi1, phi2, theta1, theta2 = -40.0, 75.0, -10.0, -10.0 # какие здесь тетта и фи, вдоль каких осей ориентированы? 
    
    D_x, D_y, D_z = tensor_output_shape
    # Убедимся, что проекция — бинарная (0 или 1)
    #projection = (projection > 0).float()  # На случай, если остались небольшие значения
    # Масштабируем проекцию до размеров D_x, D_y

    projection1 = zoom(projection1.cpu(), (D_x / projection1.shape[0], D_y / projection1.shape[1]), order=1)
    projection2 = zoom(projection2.cpu(), (D_x / projection2.shape[0], D_y / projection2.shape[1]), order=1)

    projection1=torch.tensor(projection1).to(projection1.device)
    projection2=torch.tensor(projection2).to(projection2.device)

    T = create_output_model(correct_3d_geometry.to(projection1.device), volume_shape = tensor_output_shape)
    return T
T = get_3d_tensors(sample[0][0].cuda(),sample[0][4].cuda(),sample[1].cuda())



In [ ]:
def visualize_3d_volume(volume, title="3D Reconstruction"):
    nonzero_indices = (volume > 0.01).nonzero(as_tuple=False)
    if len(nonzero_indices) > 0:
        if len(nonzero_indices) > 10000:
            indices = torch.randperm(len(nonzero_indices))[:10000]
            nonzero_indices = nonzero_indices[indices]
        values = volume[nonzero_indices[:, 0], nonzero_indices[:, 1], nonzero_indices[:, 2]]
        x_coords = nonzero_indices[:, 0].tolist()
        y_coords = nonzero_indices[:, 1].tolist()
        z_coords = nonzero_indices[:, 2].tolist()
        radii = values.tolist()
        fig = go.Figure(data=[go.Scatter3d(
            x=x_coords, y=y_coords, z=z_coords, mode='markers',
            marker=dict(size=3, color=radii, colorscale='Viridis', opacity=0.8, colorbar=dict(title="Value"))
        )])
        fig.update_layout(
            title=title,
            scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode='cube'),
            width=700, height=700,
            template="plotly_dark"
        )
        fig.show()
    else:
        print(f"No significant nonzero values (> 0.001) in {title} to visualize!")

# Визуализация
print("Visualizing target_3d...")
visualize_3d_volume(T.squeeze().squeeze(), title="Original 3D Target")


## **3D + Backprojections**

In [ ]:
def visualize_angiograph(projection1, projection2, theta, phi, correct_3d_geometry, volume_shape=(256, 256, 256)):
    vol1, v_source1, v_sensor1, sensor_points1, directions1, proj_binary1 = create_shadow_cone(projection1, theta[0], phi[0], volume_shape)
    vol2, v_source2, v_sensor2, sensor_points2, directions2, proj_binary2 = create_shadow_cone(projection2, theta[1], phi[1], volume_shape)

    backprojection = (vol1 > 0) & (vol2 > 0)
    backprojection = backprojection.float()
    print(f"Non-zero voxels in backprojection: {(backprojection > 0).sum().item()}")

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter3d(
        x=[v_source1[0].item(), v_source2[0].item()],
        y=[v_source1[1].item(), v_source2[1].item()],
        z=[v_source1[2].item(), v_source2[2].item()],
        mode='markers',
        marker=dict(size=10, color='red'),
        name='Sources'
    ))

    sensor_size = 128 * 1.4 / 2 
    for i, (v_sensor, projection) in enumerate([(v_sensor1, proj_binary1), (v_sensor2, proj_binary2)]):
        x = [v_sensor[0].item() - sensor_size, v_sensor[0].item() + sensor_size]
        y = [v_sensor[1].item() - sensor_size, v_sensor[1].item() + sensor_size]
        z = [v_sensor[2].item(), v_sensor[2].item()]
        fig1.add_trace(go.Surface(
            x=x,
            y=y,
            z=np.array(z)[None, :],
            surfacecolor=projection.cpu().numpy(),
            colorscale='Gray',
            showscale=False,
            opacity=0.7,
            name=f'Sensor {i+1}'
        ))

    num_rays = 50
    for i in range(min(num_rays, len(sensor_points1))):
        start = sensor_points1[i].cpu().numpy()
        end = v_source1.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='blue', width=2),
            name='Ray 1' if i == 0 else None,
            showlegend=(i == 0)
        ))
    for i in range(min(num_rays, len(sensor_points2))):
        start = sensor_points2[i].cpu().numpy()
        end = v_source2.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='green', width=2),
            name='Ray 2' if i == 0 else None,
            showlegend=(i == 0)
        ))

    fig1.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="Angiograph: Sensors, Sources, and Rays"
    )
    fig1.show()

    fig2 = go.Figure()

    D_x, D_y, D_z = volume_shape
    non_zero = torch.where(backprojection > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy() * (300.0 / (D_x - 1)) - 150.0
        y = non_zero[1][indices].cpu().numpy() * (300.0 / (D_y - 1)) - 150.0
        z = non_zero[2][indices].cpu().numpy() * (400.0 / (D_z - 1)) - 300.0
    else:
        x = non_zero[0].cpu().numpy() * (300.0 / (D_x - 1)) - 150.0
        y = non_zero[1].cpu().numpy() * (300.0 / (D_y - 1)) - 150.0
        z = non_zero[2].cpu().numpy() * (400.0 / (D_z - 1)) - 300.0

    fig2.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=2, color='purple', opacity=0.5),
        name='Reconstruction'
    ))

    fig2.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="3D Reconstruction"
    )
    fig2.show()

    fig3 = go.Figure()

    backprojection_resized = F.interpolate(backprojection.unsqueeze(0).unsqueeze(0), size=(128, 128, 128), mode='nearest').squeeze()

    non_zero = torch.where(backprojection_resized > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy()
        y = non_zero[1][indices].cpu().numpy()
        z = non_zero[2][indices].cpu().numpy()
    else:
        x = non_zero[0].cpu().numpy()
        y = non_zero[1].cpu().numpy()
        z = non_zero[2].cpu().numpy()

    x_mm = x * (300.0 / 127) - 150.0
    y_mm = y * (300.0 / 127) - 150.0
    z_mm = z * (400.0 / 127) - 300.0

    fig3.add_trace(go.Scatter3d(
        x=x_mm, y=y_mm, z=z_mm,
        mode='markers',
        marker=dict(size=2, color='purple', opacity=0.5),
        name='Reconstruction'
    ))

    # Оригинальная 3D-геометрия (T)
    T = get_3d_tensors(projection1, projection2, correct_3d_geometry, tensor_output_shape=[128, 128, 128])
    nonzero_indices = (T > 0.01).nonzero(as_tuple=False)
    if len(nonzero_indices) > max_points:
        indices = torch.randperm(len(nonzero_indices))[:max_points]
        nonzero_indices = nonzero_indices[indices]
    x_coords = nonzero_indices[:, 0].cpu().numpy()
    y_coords = nonzero_indices[:, 1].cpu().numpy()
    z_coords = nonzero_indices[:, 2].cpu().numpy()

    # Поворот T на 90° вокруг X-оси: Z → Y, Y → -Z
    x_rotated = x_coords
    y_rotated = z_coords
    z_rotated = -y_coords

    # Дополнительный поворот на 180° вокруг X-оси: Y → -Y, Z → -Z
    x_final_rotated = -z_coords
    y_final_rotated = x_coords
    z_final_rotated = y_coords
 

    # Масштабирование T, чтобы соответствовать размеру backprojection
    # backprojection: X: 70 мм, Y: 35 мм, Z: 200 мм
    # T: X, Y, Z: 0..127 индексов, но после create_output_model масштаб меньше
    # Оценим реальный диапазон T после поворота
    x_range = x_final_rotated.max() - x_final_rotated.min()
    y_range = y_final_rotated.max() - y_final_rotated.min()
    z_range = z_final_rotated.max() - z_final_rotated.min()

    scale_x = 70.0 / x_range if x_range != 0 else 1.0
    scale_y = 35.0 / y_range if y_range != 0 else 1.0
    scale_z = 200.0 / z_range if z_range != 0 else 1.0

    x_scaled = x_final_rotated * scale_x
    y_scaled = y_final_rotated * scale_y
    z_scaled = z_final_rotated * scale_z

    # Центрируем T относительно backprojection
    x_center = (x_mm.max() + x_mm.min()) / 2
    y_center = (y_mm.max() + y_mm.min()) / 2
    z_center = (z_mm.max() + z_mm.min()) / 2

    x_final = x_scaled + x_center - (x_scaled.max() + x_scaled.min()) / 2
    y_final = y_scaled + y_center - (y_scaled.max() + y_scaled.min()) / 2
    z_final = z_scaled + z_center - (z_scaled.max() + z_scaled.min()) / 2
# Масштабирование T, чтобы соответствовать размеру backprojection
    x_range = x_final_rotated.max() - x_final_rotated.min()
    y_range = y_final_rotated.max() - y_final_rotated.min()
    z_range = z_final_rotated.max() - z_final_rotated.min()

    scale_x = 70.0 / x_range if x_range != 0 else 1.0
    scale_y = 35.0 / y_range if y_range != 0 else 1.0
    scale_z = 200.0 / z_range if z_range != 0 else 1.0

    x_scaled = x_final_rotated * scale_x
    y_scaled = y_final_rotated * scale_y
    z_scaled = z_final_rotated * scale_z

    # Уточнённое центрирование T относительно backprojection
    x_center = (x_mm.max() + x_mm.min()) / 2
    y_center = (y_mm.max() + y_mm.min()) / 2
    z_center = (z_mm.max() + z_mm.min()) / 2

    x_final = x_scaled + x_center - (x_scaled.max() + x_scaled.min()) / 2
    y_final = y_scaled + y_center - (y_scaled.max() + y_scaled.min()) / 2
    z_final = z_scaled + z_center - (z_scaled.max() + z_scaled.min()) / 2

    fig3.add_trace(go.Scatter3d(
        x=x_final, y=y_final, z=z_final,
        mode='markers',
        marker=dict(size=2, color='green', opacity=0.5),
        name='Original 3D Target (Rotated & Scaled)'
    ))

    fig3.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="Comparison: Reconstruction vs Original 3D Target"
    )
    fig3.show()

# # Трансформация
# images_transform = T.Compose([
#     T.ToTensor(),
#     T.Resize((128, 128)),
#     T.Lambda(lambda x: x * 5.0),
# ])

# data = GeneratedDataset(images_dir, labels_dir, info_dir, images_transform=images_transform)
# sample_index = 0
# sample = data[sample_index]

projection1 = sample[0][0].cuda()
projection2 = sample[0][4].cuda()
theta = sample[2]
phi = [90 + sample[3][0], 90 + sample[3][1]]
correct_3d_geometry = sample[1].cuda()

print(f"Using angles: theta={theta}, phi={phi}")

visualize_angiograph(projection1, projection2, theta, phi, correct_3d_geometry, volume_shape=[256, 256, 256])


## **Dataset**

In [ ]:
volume_shape = [128,128,128]
vol1, v_source1, v_sensor1, sensor_points1, directions1, proj_binary1 = create_shadow_cone(projection1, theta[0], phi[0], volume_shape )
vol2, v_source2, v_sensor2, sensor_points2, directions2, proj_binary2 = create_shadow_cone(projection2, theta[1], phi[1], volume_shape )
D_x, D_y, D_z = volume_shape
# Пересечение теней
backprojection = (vol1 > 0) & (vol2 > 0)
backprojection = backprojection.float()
non_zero = torch.where(backprojection > 0)
max_points = 5000
if len(non_zero[0]) > max_points:
    indices = torch.randperm(len(non_zero[0]))[:max_points]
    x = non_zero[0][indices].cpu().numpy() 
    y = non_zero[1][indices].cpu().numpy() 
    z = non_zero[2][indices].cpu().numpy() 
else:
    x = non_zero[0].cpu().numpy() 
    y = non_zero[1].cpu().numpy() 
    z = non_zero[2].cpu().numpy() 

In [ ]:
dataset = []
for i in range(len(x)):
    a = []
    a.append(float(x[i]))
    a.append(float(y[i]))
    a.append(float(z[i]))
    dataset.append(a)
min(y)